# Regime · 02 — Threshold re-tune (F1-optimal)

**Primary interface** for exploring a regime's combined-model decision
threshold. This notebook calls `channel_heads.*` only:

- `channel_heads.eval` — `outlet_group_holdout`, `f1_optimal_threshold`,
  `classification_metrics`
- `channel_heads.inference` — `load_feature_columns`, `load_xgb_model`,
  `predict_with_threshold`

It is **read-only**: it compares the current threshold against the F1-optimal one
on the held-out test set and prints the result. It does **not** overwrite the
threshold file — that side effect belongs to the batch wrapper.

> Uses the regime model artifacts under `models/` (git-ignored). If the chosen
> regime's artifacts are absent, the cell reports that and stops gracefully.

In [1]:
import pandas as pd

from channel_heads.io.paths import PROJECT_ROOT
from channel_heads.eval import (
    classification_metrics,
    f1_optimal_threshold,
    outlet_group_holdout,
)
from channel_heads.models.xgboost import (
    load_feature_columns,
    load_xgb_model,
    predict_with_threshold,
)
from channel_heads.regimes import REGIMES

REGIME_NAME = "regB"
regime = REGIMES[REGIME_NAME]

CSV = PROJECT_ROOT / "data/results" / f"master_dataset_{regime.name}_with_emb.csv"
XGB_PATH = PROJECT_ROOT / f"models/xgb_geom_plus_cnn_emb_{regime.name}.json"
FEAT_PATH = PROJECT_ROOT / f"models/feature_columns_geom_plus_cnn_emb_{regime.name}.txt"
THR_PATH = PROJECT_ROOT / f"models/optimal_threshold_geom_plus_cnn_emb_{regime.name}.txt"

artifacts = [CSV, XGB_PATH, FEAT_PATH, THR_PATH]
have_all = all(p.exists() for p in artifacts)
print(f"regime={regime.name}  inputs present={have_all}")

regime=regB  inputs present=True


## Hold out whole outlets + score the test set — via the package

In [2]:
if not have_all:
    missing = [str(p.relative_to(PROJECT_ROOT)) for p in artifacts if not p.exists()]
    print("Artifacts absent — skipping. Missing:")
    for m in missing:
        print("  -", m)
    print(f"Run: python scripts/retune_threshold_regime.py --regime {regime.name}")
else:
    feats = load_feature_columns(FEAT_PATH)
    df = pd.read_csv(CSV)
    _, test_idx = outlet_group_holdout(df)
    df_test = df.iloc[test_idx]
    y_test = df_test["y"].astype(int).to_numpy()

    model = load_xgb_model(XGB_PATH)
    proba, _ = predict_with_threshold(model, df_test, feats, threshold=0.5)
    print(f"test n={len(y_test)} (pos={int(y_test.sum())}), features={len(feats)}")

test n=1609 (pos=510), features=9


## Current vs F1-optimal threshold (no files written)

In [3]:
if have_all:
    old_threshold = float(THR_PATH.read_text().strip().splitlines()[0])
    f1_threshold, f1_max = f1_optimal_threshold(y_test, proba)

    old_m = classification_metrics(y_test, proba, old_threshold)
    new_m = classification_metrics(y_test, proba, f1_threshold)

    print(f"  ROC AUC = {new_m['roc_auc']:.4f}   PR AUC = {new_m['pr_auc']:.4f}")
    print(f"  CURRENT threshold {old_threshold:.4f}: "
          f"P={old_m['precision']:.3f} R={old_m['recall']:.3f} F1={old_m['f1']:.3f}")
    print(f"  F1-OPTIMAL        {f1_threshold:.4f}: "
          f"P={new_m['precision']:.3f} R={new_m['recall']:.3f} F1={f1_max:.3f}")
    print("\n(read-only — threshold file not modified)")

  ROC AUC = 0.8759   PR AUC = 0.7641
  CURRENT threshold 0.7607: P=0.780 R=0.508 F1=0.615
  F1-OPTIMAL        0.5482: P=0.645 R=0.794 F1=0.712

(read-only — threshold file not modified)


---
Persist the re-tuned threshold (writes the threshold file + metrics CSV):

```bash
python scripts/retune_threshold_regime.py --regime regB
```